# Repo

> Commits, refs, diffs, and status as live Python objects

This is the literate source for `fastgit.repo`. The base `Git` class runs any git command and hands back whatever git printed, which is all a one-off call needs. For real work we want the things git talks *about*, so this module gives each one a class: `Commit`, `Ref`, `Status`, `Diff`, and friends. Each displays the way a terminal would show it, collections get their own classes and reprs, and methods chain.

In [ ]:
#| default_exp repo

## Setup

In [ ]:
#| export
from fastcore.utils import *
from fastgit.core import *
import re, subprocess
from datetime import datetime, timezone, timedelta

In [ ]:
import tempfile, shutil
from fastcore.test import test_eq

We start with an empty subclass, and use `@patch` to add each piece next to its explanation and tests. Since `Repo` is a `Git`, every raw verb keeps working on it unchanged.


In [ ]:
#| export
class Repo(Git):
    "A `Git` handle whose common queries return live objects instead of strings"

We need a repo to play with. `init` and `config` here are ordinary `Git` verb dispatch: any attribute becomes a git subcommand, and the answer comes back as a string.


In [ ]:
tmp = Path(tempfile.mkdtemp())
r = Repo(tmp)
r.init(b='main')
r.config('user.name', 'fastgit')
r.config('user.email', 'fastgit@example.com')
r.exists

True

## Commits

git will answer almost any question about a commit if you ask with a `--format` string, much as tmux answers `-F '#{...}'` queries. So we declare the fields we want once, in `_commit_f`. `_fmt` renders that spec as the `--format=` argument, and `_parse` types the reply back into one dict per record. The `%x1f`/`%x1e` separators are control characters, so they never collide with the values.


In [ ]:
#| export
_commit_f = dict(sha='%H', short='%h', parents='%P', author='%an', email='%ae', date='%aI', msg='%s')

def _fmt(fields): return '%x1f'.join(fields.values())+'%x1e'

def _conv(k, v):
    if k=='parents': return v.split()
    if k=='date': return datetime.fromisoformat(v)
    return v

def _parse(fields, s):
    "Parse `%x1f`/`%x1e`-delimited output into one dict per record"
    return [{k:_conv(k,v) for k,v in zip(fields, rec.strip('\n').split('\x1f'))}
        for rec in (s or '').split('\x1e') if rec.strip()]

In [ ]:
test_eq(_fmt(dict(sha='%H', msg='%s')), '%H%x1f%s%x1e')
rec = 'abc\x1fa\x1f\x1fme\x1fm@e\x1f2026-07-23T10:00:00+00:00\x1fhi\x1e\n'
p = _parse(_commit_f, rec*2)
test_eq(len(p), 2)
test_eq((p[0]['parents'], p[0]['msg']), ([], 'hi'))
p[0]['date']

datetime.datetime(2026, 7, 23, 10, 0, tzinfo=datetime.timezone.utc)

A commit is content-addressed, so unlike most live handles a `Commit` can never go stale. Its repr is the `--oneline` row you'd see in a terminal, and `Commits` shows a list of them as `git log --oneline` would.


In [ ]:
#| export
class GitObj(AttrDict):
    "Base for git handles: plain-text display, shown via `__repr__`"
    def _repr_markdown_(self): return None
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

class Commit(GitObj):
    "A git commit: immutable, addressed by `sha`"
    def __repr__(self): return f'{self.short} {self.msg}'

class Commits(L):
    "Commits shown as a `--oneline`-style log"
    def __repr__(self): return '\n'.join(repr(o) for o in self)
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

In [ ]:
#| export
@patch
def log(self:Repo, *args, **kwargs):
    "`Commits` for `git log` `args`: revision ranges ('main..feat'), flags, paths via `__`"
    res = self('log', *args, format=_fmt(_commit_f), mute_errors=True, **kwargs)
    return Commits(Commit(d, _g=self) for d in _parse(_commit_f, res))

@patch
def at(self:Repo, rev):
    "The `Commit` for commit-ish `rev` (sha, ref name, 'HEAD~2', `Ref`, ...); None if unknown"
    return first(self.log(str(getattr(rev, 'sha', rev)), n=1))

@patch(as_prop=True)
def head(self:Repo):
    "The `Commit` at HEAD"
    return self.at('HEAD')

@patch(as_prop=True)
def parent(self:Commit):
    "First-parent `Commit`, or None for a root commit"
    return self._g.at(self.parents[0]) if self.parents else None

Let's make some history to query. We use the raw `commit` verb for now; a friendlier `commit` arrives with the write ops later.


In [ ]:
(tmp/'a.txt').write_text('hello\nworld\n')
r.add('.')
r.commit(m='add a')
(tmp/'b.txt').write_text('data\n')
r.add('.')
r.commit(m='add b')
r.log()

d3f38e4 add b
08c95dc add a

In [ ]:
test_eq(r.log().attrgot('msg'), ['add b', 'add a'])
c1 = r.at('HEAD~1')
test_eq(c1.msg, 'add a')
test_eq(r.head.parent.sha, c1.sha)
assert c1.parent is None
test_eq(r.log('HEAD~1..')[0].msg, 'add b')
r.head

d3f38e4 add b

## Refs

A branch is a pointer: a file under `refs/heads` holding one sha. Everything else you think of as "the branch" is ancestry walked from that sha, which `log` already handles. `for-each-ref` queries pointers the way `log --format` queries commits, so the parsing is shared, and a `Ref` reprs as the line `git branch -vv` would print, tracking summary included.


In [ ]:
#| export
_ref_f = dict(name='%(refname:short)', sha='%(objectname)', upstream='%(upstream:short)',
    track='%(upstream:track)', cur='%(HEAD)', msg='%(subject)')

class Ref(GitObj):
    "A git ref: a mutable named pointer to a commit"
    def __repr__(self):
        up = f" [{self.upstream}{': '+self.track.strip('[]') if self.track else ''}]" if self.upstream else ''
        return f"{'*' if self.cur=='*' else ' '} {self.name} {self.sha[:7]}{up} {self.msg}"
    @property
    def ahead(self): return int(m.group(1)) if (m:=re.search(r'ahead (\d+)', self.track)) else 0
    @property
    def behind(self): return int(m.group(1)) if (m:=re.search(r'behind (\d+)', self.track)) else 0
    @property
    def commit(self): return self._g.at(self.sha)

class Refs(L):
    "Refs shown as `branch -vv`-style lines"
    def __repr__(self): return '\n'.join(repr(o) for o in self)
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

In [ ]:
#| export
@patch
def _refs(self:Repo, pat):
    res = self('for-each-ref', pat, format='%1f'.join(_ref_f.values())+'%1e', mute_errors=True)
    return Refs(Ref(d, _g=self) for d in _parse(_ref_f, res))

@patch(as_prop=True)
def branches(self:Repo):
    "Local branches as `Refs`"
    return self._refs('refs/heads')

@patch(as_prop=True)
def tags(self:Repo):
    "Tags as `Refs`"
    return self._refs('refs/tags')

In [ ]:
r.branch('feat')
r.tag('v0.1')
r.branches

  feat d3f38e4 add b
* main d3f38e4 add b

In [ ]:
test_eq(r.branches.attrgot('name').sorted(), ['feat', 'main'])
test_eq(first(o for o in r.branches if o.cur=='*').name, 'main')
test_eq(r.tags[0].commit.sha, r.head.sha)
test_eq(Ref(track='[ahead 2, behind 1]').ahead, 2)
test_eq(Ref(track='').behind, 0)

## Diffs

Commits hold snapshots, so a patch is something git computes by comparing two trees. Since `b = a + patch`, we spell that computation `b - a`, and `__sub__` runs `git diff a b`. The result shows file rows like `--stat` does, and keeps the patch text itself one property away. Ranges in git's own syntax pass straight through `Repo.diff`, so `r.diff('a...b')` works as at the CLI.


In [ ]:
#| export
class DiffFile(GitObj):
    "One changed file; `adds`/`dels` are None for binary files"
    def __repr__(self): return f"{self.path} | " + ('bin' if self.adds is None else f'+{self.adds} -{self.dels}')

class Diff(L):
    "Changed files shown `--stat`-style; the full patch text is in `.patch`"
    def __repr__(self):
        tot = f"{len(self)} files changed, +{sum(o.adds or 0 for o in self)} -{sum(o.dels or 0 for o in self)}"
        return '\n'.join([repr(o) for o in self] + [tot])
    def _repr_pretty_(self, p, cycle): p.text(repr(self))
    @property
    def patch(self):
        "Full patch text, as `git diff` prints it"
        return self._g('diff', *self._args, **self._kw)

In [ ]:
#| export
@patch
def diff(self:Repo, *args, **kwargs):
    "`Diff` for `git diff` `args`: commits, ranges ('a...b'), paths via `__`"
    res = self('diff', '--numstat', *args, mute_errors=True, **kwargs)
    def _row(ln):
        a,d,p = ln.split('\t', 2)
        return DiffFile(path=p, adds=None if a=='-' else int(a), dels=None if d=='-' else int(d))
    df = Diff(_row(o) for o in (res or '').splitlines())
    df._g,df._args,df._kw = self,args,kwargs
    return df

@patch
def __sub__(self:Commit, other):
    "`b - a` is the patch taking `a` to `b`, i.e. `git diff a b`"
    return self._g.diff(getattr(other, 'sha', str(other)), self.sha)

@patch
def __sub__(self:Ref, other): return self.commit - other

In [ ]:
(tmp/'a.txt').write_text('hello\nthere\nworld\n')
r.add('.')
r.commit(m='tweak a')
d = r.head - c1
d

a.txt | +1 -0
b.txt | +1 -0
2 files changed, +2 -0

In [ ]:
test_eq(len(d), 2)
test_eq((d[0].path, d[0].adds, d[0].dels), ('a.txt', 1, 0))
assert 'there' in d.patch
test_eq(len(r.diff(f'{c1.sha}...HEAD')), 2)

## Status

Status compares three things at once: HEAD's tree, the index, and the working directory, plus any files git isn't tracking at all. `--porcelain=v2` reports all of it in one stable format, so one entry class covers every case. The `xy` codes are v2's, where `.` marks the unchanged side: `.M` is modified but unstaged, `M.` staged, `??` untracked, `UU` conflicted. Unmerged entries also carry `stages`, the base/ours/theirs blob shas, which we'll need for conflict resolution shortly.


In [ ]:
#| export
class StatusEntry(GitObj):
    "One changed path; `xy` is the porcelain staged/unstaged code pair"
    def __repr__(self): return f'{self.xy} {self.path}'


def _st_entry(ln):
    t,rest = ln[0],ln[2:]
    if t in '?!': return StatusEntry(xy=t*2, path=rest)
    p = rest.split(' ')
    if t=='1': return StatusEntry(xy=p[0], path=' '.join(p[7:]))
    if t=='2':
        path,orig = ' '.join(p[8:]).split('\t')
        return StatusEntry(xy=p[0], path=path, orig=orig)
    if t=='u': return StatusEntry(xy=p[0], path=' '.join(p[9:]), stages=p[6:9])

class Status(L):
    "Working-tree state shown `status -sb`-style: branch line, entries, in-progress op"
    branch=upstream=oid=op=None
    ahead=behind = 0
    def __repr__(self):
        ab = f' [ahead {self.ahead}, behind {self.behind}]' if self.ahead or self.behind else ''
        up = f'...{self.upstream}' if self.upstream else ''
        res = [f'## {self.branch}{up}{ab}'] + [repr(o) for o in self]
        if self.op: res.append(f'# {self.op} in progress')
        return '\n'.join(res)
    def _repr_pretty_(self, p, cycle): p.text(repr(self))
    @property
    def clean(self): return not len(self)
    @property
    def conflicts(self):
        "Unmerged entries only"
        return L(o for o in self if o.xy=='UU')

Run `git status` in a terminal mid-merge and it says so; the paused operation is part of what status means. We read the same state the porcelain does, from the marker files and rebase directories inside `.git`.

In [ ]:
#| export
_op_f = dict(MERGE_HEAD='merge', CHERRY_PICK_HEAD='cherry-pick', REVERT_HEAD='revert', BISECT_LOG='bisect')

@patch(as_prop=True)
def status(self:Repo):
    "Current `Status`: branch info, changed/untracked/unmerged entries, any in-progress op"
    info,st = {},Status()
    for ln in (self('status', '--porcelain=v2', '--branch', mute_errors=True) or '').splitlines():
        if ln.startswith('# branch.'):
            k,_,v = ln[9:].partition(' ')
            info[k] = v
        elif ln: st.append(_st_entry(ln))
    st.branch,st.upstream,st.oid = info.get('head'),info.get('upstream'),info.get('oid')
    ab = re.findall(r'\d+', info.get('ab', ''))
    st.ahead,st.behind = map(int, ab) if ab else (0, 0)
    gd = Path(self('rev-parse', '--git-dir', mute_errors=True) or '.git')
    if not gd.is_absolute(): gd = self.d/gd
    st.op = first(v for k,v in _op_f.items() if (gd/k).exists())
    rb = first(p for o in ('rebase-merge','rebase-apply') if (p:=gd/o).exists())
    if not st.op and rb:
        nm = (rb/'head-name').read_text().strip().removeprefix('refs/heads/') if (rb/'head-name').exists() else ''
        st.op = f'rebase of {nm}' if nm else 'rebase'
    return st

In [ ]:
(tmp/'a.txt').write_text('hello\nthere\nworld!\n')
(tmp/'new.txt').write_text('untracked\n')
st = r.status
st

## main
.M a.txt
?? new.txt

In [ ]:
test_eq(sorted(o.xy for o in st), ['.M', '??'])
assert not st.clean
r.add('-A')
r.commit(m='more')
assert r.status.clean

## Write ops

An op that changes the working tree can pause halfway: a merge or rebase that hits a conflict stops and waits for a resolution, and git considers that normal. So nothing here raises on conflict. `merge`, `rebase`, `pull`, and `stash` always return the resulting `Status`, clean or paused, and reading it tells you which you got. `commit` can't pause, so it returns the new head `Commit` instead.


In [ ]:
#| export
@patch
def commit(self:Repo, msg=None, *args, **kwargs):
    "Commit staged changes, returning the new head `Commit`"
    if msg: kwargs['m'] = msg
    self('commit', *args, **kwargs)
    return self.head

@patch
def merge(self:Repo, *args, **kwargs):
    "Merge, returning the resulting `Status`: a conflict is a state, not an error"
    self('merge', *args, mute_errors=True, **kwargs)
    return self.status

@patch
def rebase(self:Repo, *args, **kwargs):
    "Rebase, returning the resulting `Status`"
    self('rebase', *args, mute_errors=True, **kwargs)
    return self.status

@patch
def pull(self:Repo, *args, **kwargs):
    "Pull, returning the resulting `Status`"
    self('pull', *args, mute_errors=True, **kwargs)
    return self.status

`feat` still points where `main` was two commits ago, and both lines have since edited the same region of `a.txt`, so merging it conflicts. The returned status shows the unmerged path and the paused merge:


In [ ]:
r.switch('feat')
(tmp/'a.txt').write_text('hello\nfeat\nworld\n')
r.add('.')
r.commit('feat change')
r.switch('main')
st = r.merge('feat')
st

## main
UU a.txt
# merge in progress

In [ ]:
test_eq(st.op, 'merge')
assert not st.clean
test_eq(st[0].xy, 'UU')
test_eq(len(st[0].stages), 3)
test_eq(st.conflicts, [st[0]])

In [ ]:
#| export
@patch
def cat(self:Repo, spec):
    "Exact file content at `spec` ('rev:path', or ':1:'/':2:'/':3:' + path for conflict stages); no stripping, unlike raw verbs"
    args = [*(self.pre or []), 'git', '-C', str(self.d), 'show', str(spec)]
    return subprocess.run(args, capture_output=True, text=True, check=True).stdout

To resolve, we need each side's exact content. During a conflict git exposes the three versions as `:1:path` (base), `:2:path` (ours), and `:3:path` (theirs), and `cat` reads them byte-for-byte. `show` would almost work, but `callgit` strips trailing whitespace from every raw verb's reply, which is right for a terminal and wrong for a file: it would eat the final newline. With `st.conflicts` narrowing to the unmerged entries, resolution is ordinary git: write the file, `add` it, `commit`. The two parents on the new head prove the merge concluded.


In [ ]:
test_eq(r.cat(':2:a.txt'), 'hello\nthere\nworld!\n')
test_eq(r.cat(':3:a.txt'), 'hello\nfeat\nworld\n')
(tmp/'a.txt').write_text('hello\nthere\nfeat\nworld!\n')
r.add('a.txt')
mc = r.commit('merge feat')
test_eq(len(mc.parents), 2)
assert r.status.clean
mc

66b7277 merge feat

`rebase` keeps the same contract. `feat` itself is already merged, so rebasing it would fast-forward; instead we cut a branch from before the merge, edit the same region again, and replay it onto `main`. Mid-rebase, `op` names the branch being replayed, and `--abort` backs out cleanly.


In [ ]:
r.switch('-c', 'topic', 'HEAD~1')
(tmp/'a.txt').write_text('hello\nTOPIC\nworld!\n')
r.add('.')
r.commit('topic change')
st = r.rebase('main')
st

## (detached)
UU a.txt
# rebase of topic in progress

In [ ]:
test_eq(st.op, 'rebase of topic')
assert not st.clean
assert r.rebase('--abort').clean
test_eq(r.current_branch, 'topic')
r.switch('main')
assert r.status.clean

## Blame

`blame` answers "which commit last touched each line?". `--line-porcelain` is the stable form, one metadata block per line, and each row keeps its commit's sha, so the full `Commit` is one hop away. The repr is the terminal's: sha, author, date, line number, content. The `-L` range forms become keyword arguments, built by `_lspec`: `lines=(start,end)`, `func='name'` (git's `:funcname` form), and `regex=` (a content pattern, or a `(start, end)` pair).

In [ ]:
#| export
def _lbound(o):
    "One `-L` range bound: an int line number, a `+N`/`-N` offset str, or a /pattern/"
    if isinstance(o, int): return str(o)
    if re.fullmatch(r'[+-]\d+', o): return o
    return '/'+o.replace('/', r'\/')+'/'

def _lspec(func=None, lines=None, regex=None):
    "A `git -L` range spec from whichever of `func`, `lines`, or `regex` is given"
    if func: return f':{func}'
    if lines: return f'{lines[0]},{lines[1]}'
    if regex:
        if isinstance(regex, str): regex = (regex, '+1')  # bare pattern: just the matched line
        return ','.join(_lbound(o) for o in regex)

def _tz(s): return timezone(timedelta(minutes=(-1 if s[0]=='-' else 1)*(int(s[1:3])*60+int(s[3:5]))))

class BlameLine(GitObj):
    "One blamed line; `sha`/`author`/`date`/`msg` describe the last commit to touch it"
    def __repr__(self): return f'{self.sha[:7]} ({self.author} {self.date:%Y-%m-%d %H:%M} {self.lineno:>3}) {self.line}'
    @property
    def commit(self): return self._g.at(self.sha)

class Blame(L):
    "Blame lines shown as `git blame` shows them"
    def __repr__(self): return '\n'.join(repr(o) for o in self)
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

@patch
def blame(self:Repo, path, *args, func=None, lines=None, regex=None):
    "`Blame` for `path` (`args` may add a revision or flags); `func`/`lines`/`regex` pick a `-L` range"
    spec = _lspec(func, lines, regex)
    cmd = [*(self.pre or []), 'git', '-C', str(self.d), 'blame', '--line-porcelain',
        *(['-L', spec] if spec else []), *args, '--', str(path)]
    cur,res = {},Blame()
    for ln in subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.splitlines():
        if ln.startswith('\t'): res.append(BlameLine(cur, line=ln[1:], _g=self))
        else:
            k,_,v = ln.partition(' ')
            if re.fullmatch(r'[0-9a-f]{40}', k): cur = dict(sha=k, lineno=int(v.split()[1]))
            elif k=='author': cur['author'] = v
            elif k=='author-time': cur['ts'] = int(v)
            elif k=='author-tz': cur['date'] = datetime.fromtimestamp(cur.pop('ts'), _tz(v))
            elif k=='summary': cur['msg'] = v
    return res

Our `a.txt` is four lines from four different commits, one of them merged in from another branch, and blame sees straight through the merge:

In [ ]:
r.blame('a.txt')

08c95dc (fastgit 2026-07-24 12:11   1) hello
94fbe56 (fastgit 2026-07-24 12:11   2) there
b5c9ad8 (fastgit 2026-07-24 12:11   3) feat
d8f2453 (fastgit 2026-07-24 12:11   4) world!

In [ ]:
test_eq(r.blame('a.txt').attrgot('line'), ['hello', 'there', 'feat', 'world!'])
test_eq(r.blame('a.txt')[2].commit.msg, 'feat change')
test_eq(len({o.sha for o in r.blame('a.txt')}), 4)

`func=` finds a definition by name. git matches the name against *funcname lines*, and its default pattern only recognizes definitions at column 0, which covers top-level `def`s and `@patch` methods; to find indented methods by name too, add `*.py diff=python` to the repo's `.gitattributes` and git's Python-aware pattern takes over. `regex=` needs no configuration at all, since it matches line content directly:

In [ ]:
(tmp/'weather.py').write_text('def c2f(c):\n    return c*9/5+32\n\ndef f2c(f):\n    return (f-32)*5/9\n')
r.add('.')
r.commit('add conversions')
(tmp/'weather.py').write_text('def c2f(c):\n    "Celsius to Fahrenheit"\n    return c*9/5+32\n\ndef f2c(f):\n    return (f-32)*5/9\n')
r.add('.')
r.commit('document c2f')
r.blame('weather.py', func='c2f')

08f79e7 (fastgit 2026-07-24 12:11   1) def c2f(c):
e1c1834 (fastgit 2026-07-24 12:11   2)     "Celsius to Fahrenheit"
08f79e7 (fastgit 2026-07-24 12:11   3)     return c*9/5+32
08f79e7 (fastgit 2026-07-24 12:11   4) 

In [ ]:
b = r.blame('weather.py', func='c2f')
test_eq(b[1].commit.msg, 'document c2f')
test_eq(len(r.blame('weather.py', lines=(1,1))), 1)
test_eq(r.blame('weather.py', regex='def f2c')[0].line, 'def f2c(f):')

Blame tells you who last touched each line; `trace` tells the whole story. `git log -L` replays every commit that changed a range, patch by patch. The range keywords are the same, and the result is ordinary `Commits`, each carrying the `.patch` that changed the range:

In [ ]:
#| export
@patch
def trace(self:Repo, path, *args, func=None, lines=None, regex=None, **kwargs):
    "Trace the evolution of a range of `path` (`git log -L`): `Commits` newest first, each with its `.patch`"
    spec = _lspec(func, lines, regex)
    if not spec: raise TypeError('trace needs one of func, lines, or regex')
    res = self('log', f'-L{spec}:{path}', *args, format='%x1e'+'%x1f'.join(_commit_f.values()), mute_errors=True, **kwargs)
    out = Commits()
    for rec in (res or '').split('\x1e'):
        if not rec.strip(): continue
        head,_,patch = rec.partition('\n')
        out.append(Commit({k:_conv(k,v) for k,v in zip(_commit_f, head.split('\x1f'))}, _g=self, patch=patch.strip('\n')))
    return out

In [ ]:
t = r.trace('weather.py', func='c2f')
t

e1c1834 document c2f
08f79e7 add conversions

In [ ]:
test_eq(t.attrgot('msg'), ['document c2f', 'add conversions'])
assert 'Celsius' in t[0].patch
print(t[0].patch)
tr = r.trace('weather.py', regex='return c')             # a bare pattern traces just the matched line
test_eq(tr.attrgot('msg'), ['add conversions'])
tr = r.trace('weather.py', regex=('def c2f', '+2'))      # pair elements may be patterns, line numbers, or ±offsets
test_eq(tr.attrgot('msg'), ['document c2f', 'add conversions'])

diff --git a/weather.py b/weather.py
index 7431045..9cbb8e3 100644
--- a/weather.py
+++ b/weather.py
@@ -1,3 +1,4 @@
 def c2f(c):
+    "Celsius to Fahrenheit"
     return c*9/5+32
 


## Remotes

None of this has needed a network, and the remote ops don't either: a bare repo in another temp dir is a perfectly good `origin`. `Remote` itself is a small noun, shown the way `git remote -v` lists it.

In [ ]:
#| export
class Remote(GitObj):
    "A configured remote"
    def __repr__(self): return f'{self.name}\t{self.url}'

class Remotes(L):
    "Remotes shown as `git remote -v`-style lines"
    def __repr__(self): return '\n'.join(repr(o) for o in self)
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

@patch(as_prop=True)
def remotes(self:Repo):
    "Configured remotes as `Remotes`"
    res = {}
    for ln in (self('remote', '-v', mute_errors=True) or '').splitlines():
        nm,rest = ln.split('\t')
        res.setdefault(nm, Remote(name=nm, url=rest.split()[0], _g=self))
    return Remotes(res.values())

In [ ]:
bare = Path(tempfile.mkdtemp())/'origin.git'
Git(bare.parent)('init', '--bare', '-b', 'main', bare.name)
r.remote('add', 'origin', str(bare))
r.remotes

origin	/tmp/tmp73enby1b/origin.git

`push -u` publishes the branch and records its upstream. Rich `push` returns the current branch's refreshed `Ref`, so the new tracking bracket appears right in the output; `fetch` returns the refreshed `branches` for the same reason, since what it moves is the tracking refs.

In [ ]:
#| export
@patch
def fetch(self:Repo, *args, **kwargs):
    "Fetch, returning the local `branches` with refreshed tracking info"
    self('fetch', *args, **kwargs)
    return self.branches

@patch
def push(self:Repo, *args, **kwargs):
    "Push, returning the current branch's refreshed `Ref`"
    self('push', *args, **kwargs)
    return first(o for o in self.branches if o.cur=='*')

In [ ]:
r.push('-u', 'origin', 'main')

* main e1c1834 [origin/main] document c2f

`clone` must be a classmethod, since until it runs there is no repo to hold a handle on:

In [ ]:
#| export
@patch(cls_method=True)
def clone(cls:Repo, url, path=None, **kwargs):
    "Clone `url` into `path` (default: the repo's name in the cwd), returning a `Repo` on the checkout"
    if path is None: path = re.sub(r'\.git$', '', str(url).rstrip('/').split('/')[-1])
    path = Path(path)
    Git(path.parent)('clone', str(url), path.name, **kwargs)
    return cls(path)

In [ ]:
tmp2 = Path(tempfile.mkdtemp())/'copy'
r2 = Repo.clone(bare, tmp2)
r2.log()

e1c1834 document c2f
08f79e7 add conversions
66b7277 merge feat
d8f2453 more
b5c9ad8 feat change
94fbe56 tweak a
d3f38e4 add b
08c95dc add a

A commit pushed from the clone leaves our first checkout behind. `fetch` shows the gap, and `pull` closes it:

In [ ]:
r2.config('user.name', 'fastgit')
r2.config('user.email', 'fastgit@example.com')
(tmp2/'c.txt').write_text('remote work\n')
r2.add('.')
r2.commit('from the clone')
r2.push()
r.fetch()

  feat b5c9ad8 feat change
* main e1c1834 [origin/main: behind 1] document c2f
  topic 8e64746 topic change

In [ ]:
test_eq(first(o for o in r.branches if o.cur=='*').behind, 1)
assert r.pull().clean
test_eq(r.head.msg, 'from the clone')
test_eq(r.remotes[0].name, 'origin')

## Stashes

A stash entry is a real commit whose parents link the stashed worktree and index states; `stash@{n}` is a reflog address for it. So `Stash` subclasses `Commit`, adding the selector and the `stash list` line format.


In [ ]:
#| export
_stash_f = dict(_commit_f, sel='%gd')

class Stash(Commit):
    "A stash entry: a real commit addressed `stash@{n}`"
    def __repr__(self): return f'{self.sel}: {self.msg}'

class Stashes(L):
    "Stash entries shown as `stash list`-style lines"
    def __repr__(self): return '\n'.join(repr(o) for o in self)
    def _repr_pretty_(self, p, cycle): p.text(repr(self))

@patch(as_prop=True)
def stashes(self:Repo):
    "Stash entries, newest first"
    res = self('stash', 'list', format=_fmt(_stash_f), mute_errors=True)
    return Stashes(Stash(d, _g=self) for d in _parse(_stash_f, res))

@patch
def stash(self:Repo, msg=None, **kwargs):
    "Stash worktree changes, returning the resulting `Status`"
    if msg: kwargs['m'] = msg
    self('stash', 'push', mute_errors=True, **kwargs)
    return self.status

@patch
def pop(self:Stash, **kwargs):
    "Re-apply and drop this stash, returning the resulting `Status`"
    self._g('stash', 'pop', self.sel, mute_errors=True, **kwargs)
    return self._g.status

@patch
def drop(self:Stash, **kwargs):
    "Delete this stash, returning the resulting `Status`"
    self._g('stash', 'drop', self.sel, mute_errors=True, **kwargs)
    return self._g.status

In [ ]:
(tmp/'a.txt').write_text('wip\n')
assert r.stash('wip work').clean
r.stashes

stash@{0}: On main: wip work

In [ ]:
st = r.stashes[0].pop()
test_eq(st[0].xy, '.M')
test_eq(r.stashes, [])

In [ ]:
#| hide
for p in (tmp, bare.parent, tmp2.parent): shutil.rmtree(p)


## Export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()